# 📊 Data Profiling — Team Outliers

**Mục tiêu**: Kiểm tra tổng quan 15 file dữ liệu, phát hiện giá trị thiếu (null), kiểu dữ liệu và các vấn đề bất thường.

In [ ]:
import polars as pl
import sys
from pathlib import Path
from IPython.display import display

# Add src to path to import DataLoader
sys.path.append(str(Path("..").resolve()))
from src.data_loader import DataLoader

loader = DataLoader()
all_dfs = loader.load_all()

✅ Loaded customers: 121,930 rows
✅ Loaded geography: 39,948 rows
✅ Loaded inventory: 60,247 rows
✅ Loaded order_items: 714,669 rows
✅ Loaded orders: 646,945 rows
✅ Loaded payments: 646,945 rows
✅ Loaded products: 2,412 rows
✅ Loaded promotions: 50 rows
✅ Loaded returns: 39,939 rows
✅ Loaded reviews: 113,551 rows
✅ Loaded sales: 3,833 rows
✅ Loaded sample_submission: 548 rows
✅ Loaded shipments: 566,067 rows
✅ Loaded web_traffic: 3,652 rows


## 1. Kiểm tra số lượng dòng và cột

In [ ]:
summary = []
for name, df in all_dfs.items():
    summary.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Size (MB)": round(df.estimated_size() / (1024 * 1024), 2)
    })

pl.DataFrame(summary).sort("Rows", descending=True)

Table,Rows,Columns,Size (MB)
str,i64,i64,f64
"""order_items""",714669,7,29.9
"""orders""",646945,8,39.38
"""payments""",646945,4,20.33
"""shipments""",566067,4,12.96
"""customers""",121930,7,5.79
…,…,…,…
"""sales""",3833,3,0.07
"""web_traffic""",3652,7,0.19
"""products""",2412,8,0.14


## 2. Kiểm tra giá trị Null

In [ ]:
def check_nulls(name, df):
    null_counts = df.null_count()
    null_pct = (null_counts / len(df) * 100)
    return null_pct

for name, df in all_dfs.items():
    nulls = check_nulls(name, df)
    if nulls.sum_horizontal()[0] > 0:
        print(f"\n⚠️ {name} has null values:")
        # Filter columns with nulls > 0
        for col in df.columns:
            pct = nulls[col][0]
            if pct > 0:
                print(f"  - {col}: {pct:.2f}%")


⚠️ order_items has null values:
  - promo_id: 61.34%
  - promo_id_2: 99.97%

⚠️ promotions has null values:
  - applicable_category: 80.00%


## 3. Kiểu dữ liệu

In [ ]:
for name, df in all_dfs.items():
    print(f"\n--- {name} ---")
    print(df.schema)


--- customers ---
Schema({'customer_id': Int64, 'zip': Int64, 'city': String, 'signup_date': Date, 'gender': String, 'age_group': String, 'acquisition_channel': String})

--- geography ---
Schema({'zip': Int64, 'city': String, 'region': String, 'district': String})

--- inventory ---
Schema({'snapshot_date': Date, 'product_id': Int64, 'stock_on_hand': Int64, 'units_received': Int64, 'units_sold': Int64, 'stockout_days': Int64, 'days_of_supply': Float64, 'fill_rate': Float64, 'stockout_flag': Int64, 'overstock_flag': Int64, 'reorder_flag': Int64, 'sell_through_rate': Float64, 'product_name': String, 'category': String, 'segment': String, 'year': Int64, 'month': Int64})

--- order_items ---
Schema({'order_id': Int64, 'product_id': Int64, 'quantity': Int64, 'unit_price': Float64, 'discount_amount': Float64, 'promo_id': String, 'promo_id_2': String})

--- orders ---
Schema({'order_id': Int64, 'order_date': Date, 'customer_id': Int64, 'zip': Int64, 'order_status': String, 'payment_method':

## 4. Phân bố và thống kê 

In [ ]:
for name, df in all_dfs.items():
    print(f"\n--- {name} ---")
    num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]
    if num_cols:
        display(df.select(num_cols).describe())
    else:
        print("No numerical columns")


--- customers ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_26392\3365962957.py:3: DeprecationWarning: `NUMERIC_DTYPES` was deprecated in version 1.0.0. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]


statistic,customer_id,zip
str,f64,f64
"""count""",121930.0,121930.0
"""null_count""",0.0,0.0
"""mean""",78736.898663,50990.165595
"""std""",45492.202886,26871.914605
"""min""",1.0,1001.0
"""25%""",39343.0,28689.0
"""50%""",78785.0,49835.0
"""75%""",118157.0,73488.0
"""max""",157563.0,99950.0



--- geography ---


statistic,zip
str,f64
"""count""",39948.0
"""null_count""",0.0
"""mean""",50895.084735
"""std""",27042.257341
"""min""",1.0
"""25%""",28280.0
"""50%""",49877.0
"""75%""",73526.0
"""max""",99950.0


...

statistic,Revenue,COGS
str,f64,f64
"""count""",548.0,548.0
"""null_count""",0.0,0.0
"""mean""",3.2498e6,2.7838e6
"""std""",1.5817e6,1.3603e6
"""min""",977332.85,790258.95
"""25%""",2033376.9,1708216.3
"""50%""",2.9798e6,2.5775e6
"""75%""",4.1814e6,3.5156e6
"""max""",9.2834e6,8.3415e6



--- shipments ---


statistic,order_id,shipping_fee
str,f64,f64
"""count""",566067.0,566067.0
"""null_count""",0.0,0.0
"""mean""",415816.869664,4.962857
"""std""",240007.311562,8.887355
"""min""",1.0,0.0
"""25%""",208193.0,0.87
"""50%""",415866.0,1.73
"""75%""",623219.0,2.6
"""max""",834325.0,32.0



--- web_traffic ---


statistic,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec
str,f64,f64,f64,f64,f64
"""count""",3652.0,3652.0,3652.0,3652.0,3652.0
"""null_count""",0.0,0.0,0.0,0.0,0.0
"""mean""",25041.768072,19031.404436,108615.224535,0.004487,210.283242
"""std""",9422.609335,7237.953062,44472.055524,0.000753,63.771711
"""min""",7973.0,6136.0,30451.0,0.0032,100.1
"""25%""",17101.0,12915.0,73001.0,0.00385,156.7
"""50%""",23645.0,17925.0,101030.0,0.00445,209.3
"""75%""",31779.0,24191.0,138084.0,0.00516,266.2
"""max""",50947.0,40430.0,275560.0,0.0058,319.9


## 5. Phát hiện Outlier (Outlier Detection)

In [ ]:
def check_outliers_iqr(df, col):
    s = df.get_column(col).drop_nulls()
    if len(s) == 0:
        return 0, 0, 0
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    if q1 is None or q3 is None:
        return 0, 0, 0
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df.filter((pl.col(col) < lower_bound) | (pl.col(col) > upper_bound))
    return len(outliers), lower_bound, upper_bound

for name, df in all_dfs.items():
    num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]
    outlier_summary = []
    for col in num_cols:
        try:
            n_outliers, lb, ub = check_outliers_iqr(df, col)
            if n_outliers > 0:
                outlier_summary.append({
                    "Column": col,
                    "Outliers": n_outliers,
                    "%": round(n_outliers / len(df) * 100, 2)
                })
        except Exception as e:
            pass
    if outlier_summary:
        print(f"\n--- {name} Outliers ---")
        display(pl.DataFrame(outlier_summary).sort("Outliers", descending=True))


C:\Users\Admin\AppData\Local\Temp\ipykernel_26392\405703035.py:17: DeprecationWarning: `NUMERIC_DTYPES` was deprecated in version 1.0.0. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  num_cols = [col for col, dtype in df.schema.items() if dtype in pl.NUMERIC_DTYPES]



--- inventory Outliers ---


Column,Outliers,%
str,i64,f64
"""overstock_flag""",14305,23.74
"""days_of_supply""",7455,12.37
"""stock_on_hand""",6432,10.68
"""units_received""",6392,10.61
"""units_sold""",6388,10.6
"""sell_through_rate""",1284,2.13
"""stockout_days""",724,1.2
"""fill_rate""",724,1.2



--- order_items Outliers ---


Column,Outliers,%
str,i64,f64
"""discount_amount""",105767,14.8
"""unit_price""",8623,1.21


...

Column,Outliers,%
str,i64,f64
"""COGS""",18,3.28
"""Revenue""",16,2.92



--- shipments Outliers ---


Column,Outliers,%
str,i64,f64
"""shipping_fee""",76050,13.43



--- web_traffic Outliers ---


Column,Outliers,%
str,i64,f64
"""page_views""",18,0.49
